# Production plan with a distribution network

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/production_plan/notebooks/distribution_network.ipynb)

A company assembles electric bicycles in three plants and plans how many bikes each plant assembles each month from January to June. In the third version of the model, M2, the plants ship to a central warehouse, which holds stock and delivers the orders. With the late demand peak, a peak of the orders from April to June, M2 has no solution: the bikes that have to be assembled in advance do not fit in the plants and the central warehouse. This notebook uses the fourth version, M3: a distribution network in Spain with the plants, the central warehouse, a rail terminal, two regional warehouses and four local warehouses, which receive the orders. The network is a list of arcs, each with a transport cost and an arc capacity.

Contents of the notebook:

1. Distribution network of M3.
2. Linear model of the distribution network.
3. Optimal plan under the late demand peak.
4. A test with one datum changed: the node capacity of regional warehouse 2.
5. An analysis of the plan of M3, not a new version of the model: the minimum cost flow problem of one month.

The cells run in order, with ▶ or *Shift + Enter*. Some cells have a form (sliders and tick boxes): after changing a value, run the cell again. Outside Colab the form is a line such as `show_tables = False`: edit the value and run the cell.

The figures are interactive. Place the pointer on a bar, a node or an arc (on a tablet or a phone, touch it) to read its values. A list at the top left of a figure changes what it shows without running the cell again. The buttons that appear at the top right of a figure zoom in and out; a double click goes back to the whole figure.

The table gives the symbols of M0 to M3, with the term that names each concept, and those of the analysis of the plan of M3 in section 5. The column «From» gives the version or the analysis where each symbol first appears; in M3 the warehouses are nodes $i$ of the network.

| Symbol | Meaning | Term | Unit | From |
|---|---|---|---|---|
| $F$, $f$ | Plants | — | — | M0 |
| $T$, $t$ | Months, January to June; $t = 0$ is the end of December | — | — | M0 |
| $\tau$ | Last month of the horizon | — | — | M1 |
| $V$, $i$ | Nodes of the network: plants, warehouses and the rail terminal | — | — | M3 |
| $A$, $a$ | Arcs of the network; the arc $a = \langle i, j \rangle$ goes from its origin node $i$ to its destination node $j$ | — | — | M3 |
| $A^{\text{in}}_j$; $A^{\text{out}}_i$ | Arcs that enter node $j$; arcs that leave node $i$ | — | — | M3 |
| $V^d$ | Demand nodes: the local warehouses | demand node | — | M3 |
| $c_f$ | Unit assembly cost at plant $f$ | — | €/bike | M0 |
| $K_{ft}$ | Assembly capacity of plant $f$ in month $t$ | — | bikes/month | M0 |
| $d_{it}$ | Orders that node $i$ receives in month $t$ | — | bikes/month | M3 |
| $c^h_i$ | Holding cost at node $i$ | holding cost | €/(bike·month) | M1 |
| $h_{i0}$ | Initial stock at node $i$, at the end of December | — | bikes | M1 |
| $h^{\text{end}}_i$ | Stock that node $i$ must hold at the end of the horizon | required final stock | bikes | M1 |
| $K^h_i$ | Units node $i$ can hold | storage capacity | bikes | M2 |
| $c^b_i$ | Cost of one bike backordered for one month at node $i$ | backorder cost | €/(bike·month) | M2 |
| $b_{i0}$; $\bar b_i$ | Backorders at node $i$ at the end of December; maximum at the end of the horizon | backorders | bikes | M2 |
| $c^x_a$ | Cost of carrying one bike along arc $a$ | transport cost | €/bike | M3 |
| $K^x_a$ | Maximum flow on arc $a$ in a month | arc capacity | bikes/month | M3 |
| $K^n_i$ | Maximum flow that can enter node $i$ in a month | node capacity | bikes/month | M3 |
| $p_{ft}$ | Bikes assembled at plant $f$ in month $t$ | — | bikes/month | M0 |
| $h_{it}$ | Bikes in stock at node $i$ at the end of month $t$ | — | bikes | M1 |
| $x_{at}$ | Bikes shipped on arc $a$ in month $t$ | shipments | bikes/month | M1 |
| $s_{it}$ | Bikes that node $i$ delivers in month $t$ to orders of that month or to backorders | deliveries | bikes/month | M2 |
| $b_{it}$ | Orders received by node $i$ and not yet delivered at the end of month $t$ | backorders | bikes | M2 |
| $q_{it}$ | Net supply of node $i$ in month $t$ in the plan of M3: the bikes that the plan makes it put into the network in that month; positive at a supply node, negative at a demand node | net supply | bikes/month | Analysis of M3 (section 5) |

The [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/distribution_network.md) explains all the data and results. Running the notebook is optional: it is support material, not part of the assessment.

Original case by J. J. Ramos (UAB, 2026). The locations were obtained from Nominatim (OpenStreetMap), the road distances from OSRM and the rail distance from OpenRailRouting. The remaining data are synthetic and were generated specifically for this case study.

In [ ]:
# @title Setup { display-mode: "form" }
# In Google Colab, install the example from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/production_plan"
    %pip install -q {url}
    # A session that already had the example keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    # and unless its old code is removed from memory, so that the import below reads
    # the new one: no need to restart the session
    import importlib
    for name in [m for m in sys.modules if m == "plants" or m.startswith("plants.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
from plants import distribution_network as dn

## 1. Distribution network of M3

This section of the notebook gives the data of M3. The next cell first prints the transport rate of road and of rail, from which the transport cost of each arc was computed with its distance. Then it draws three figures. The map places each node in its city; the buttons above it show or hide the names of the cities. The second figure draws the network by stages, from the plants on the left to the local warehouses on the right, with the rail arc dashed; the list at the top chooses what is written on each arc: its transport cost, its arc capacity or its number. Place the pointer on a node or an arc to read all its data. The third figure gives the orders of each local warehouse in each month, against the assembly capacity of the three plants (dashed line).

`show_tables`: instead of the figures, four tables: the place of each node, the list of arcs, the data of each node and the orders of each local warehouse.

With the figures of the next cell, or with its tables, answer: which kind of node is each site of the network, and which of them assemble, deliver orders or hold stock?

In [ ]:
# @title Distribution network of M3 { display-mode: "form" }
show_tables = False  # @param {type:"boolean"}
dn.show_data(show_tables=show_tables)

## 2. Linear model of the distribution network

$$
\begin{aligned}
\min \quad & \sum_{t \in T} \Big( \sum_{f \in F} c_f\, p_{ft} + \sum_{i \in V} c^h_i\, h_{it} + \sum_{a \in A} c^x_a\, x_{at} + \sum_{i \in V^d} c^b_i\, b_{it} \Big) \\
\text{s.t.} \quad & h_{i,t-1} + p_{it} + \sum_{a \in A^{\text{in}}_i} x_{at} = \sum_{a \in A^{\text{out}}_i} x_{at} + s_{it} + h_{it} && \forall i \in V,\ t \in T && \text{(stock balance at each node)} \\
& \sum_{a \in A^{\text{out}}_i} x_{at} + s_{it} \le h_{i,t-1} + p_{it} + \sum_{a \in A^{\text{in}}_i} x_{at} && \forall i \in V,\ t \in T && \text{(shipment limit at each node)} \\
& b_{i,t-1} + d_{it} = s_{it} + b_{it} && \forall i \in V^d,\ t \in T && \text{(backorder balance)} \\
& x_{at} \le K^x_a && \forall a \in A,\ t \in T && \text{(arc capacity)} \\
& \sum_{a \in A^{\text{in}}_i} x_{at} \le K^n_i && \forall i \text{ with a node capacity},\ t \in T && \text{(node capacity)} \\
& h_{it} \le K^h_i && \forall i \in V,\ t \in T && \text{(storage capacity)} \\
& p_{ft} \le K_{ft} && \forall f \in F,\ t \in T && \text{(assembly capacity)} \\
& b_{i\tau} \le \bar b_i \quad \forall i \in V^d, \qquad h_{i\tau} \ge h^{\text{end}}_i \quad \forall i \in V && && \text{(backorders and required final stock at the end)} \\
& p_{ft},\ h_{it},\ x_{at},\ s_{it},\ b_{it} \ge 0
\end{aligned}
$$

with $h_{i0}$ and $b_{i0}$ given, $p_{it} = 0$ if node $i$ is not a plant and $s_{it} = 0$ if node $i$ is not a demand node. One stock balance holds at every node: what the node held at the start of the month, plus what it assembles and what arrives on the arcs that enter it, equals what leaves on the arcs that leave it, plus its deliveries, plus what it holds at the end of the month. Only the plants assemble and only the local warehouses deliver; at the rail terminal both are 0, and its storage capacity is 0. There is one flow variable for each arc of the list.

## 3. Optimal plan under the late demand peak

This section of the notebook solves M3. The next cell first prints why M2 has no solution under the late demand peak, and then the cost of the optimal plan of M3. Then it draws five figures. The first is the network with the flow on each arc in the month chosen in the list at the top: a red arc has its arc capacity full, and a light grey arc carries no flow. Place the pointer on a node to read its stock at the end of that month; a plant also gives what it assembles, a local warehouse what it delivers and its backorders, and regional warehouse 2 what it receives against its node capacity. The second gives the flow on one arc in each month against its arc capacity; the third, the bikes assembled in each month against the capacity, for one plant or for the three; the fourth, the stock at each stage of the network at the end of each month. The fifth gives the shadow price of each capacity that is full in the month chosen in its list (€/bike); a negative shadow price is a saving.

`show_tables`: instead of the figures, five tables: the bikes assembled, the flow on each arc, the stock of each node, the deliveries and backorders of the local warehouses, and the shadow prices of the full capacities.

With the figures of the next cell, or with its tables, answer:

1. At the end of which month is every storage capacity of the network full? Which of them has the largest saving in its shadow price that month, and why there?
2. In May, which capacities on the routes to regional warehouse 2 are full? For each full arc capacity, which two routes does its shadow price compare?

In [ ]:
# @title Optimal plan under the late demand peak { display-mode: "form" }
show_tables = False  # @param {type:"boolean"}
dn.network_plan(show_tables=show_tables);

## 4. Node capacity of regional warehouse 2

This section of the notebook solves M3 again with another node capacity of regional warehouse 2, the bikes it can receive in a month by its three routes; the other data are those of section 3. The next cell prints the node capacity and the cost of the optimal plan, or that there is no plan. Then it draws two figures. The first gives, for each month, the bikes that regional warehouse 2 receives by each route, stacked, against its node capacity (dashed line), and its stock at the end of the month; in its list, the routes to Valencia instead. The second gives, for each month, the orders of Seville (local warehouse 4), the bikes delivered to them and the orders still pending at the end of the month.

`node_capacity`: the node capacity of regional warehouse 2, in bikes/month; in section 3 it is 280.

`show_tables`: instead of the figures, the same data in tables.

Lower the node capacity from 300 to 200, and answer with the figures of the next cell or with its tables: when the node capacity is full, how does the plan still supply Seville, and where does Valencia get its bikes? Below which value is there no plan, and why?

In [ ]:
# @title Optimal plan for a node capacity of regional warehouse 2 { display-mode: "form" }
node_capacity = 280  # @param {type:"slider", min:200, max:300, step:10}
show_tables = False  # @param {type:"boolean"}
dn.node_capacity_test(node_capacity, show_tables=show_tables);

## 5. Minimum cost flow problem of one month

This section of the notebook takes the optimal plan of M3 of section 3, one month at a time; M3 and its plan do not change. In each month, what the plan assembles, holds and delivers at a node sets how many bikes the node puts into the network or takes out of it, its net supply: positive at a supply node, negative at a demand node and zero at a transhipment node. With these net supplies, the arc capacities and the node capacity of regional warehouse 2, the next cell solves the minimum cost flow problem of each month on its own.

The cell draws the network in the month chosen in the list at the top: each node in the colour of its kind, and the flow on each arc; an arc whose capacity is full is red, and an arc with no flow is light grey. Orange arrows that are not arcs show, at each node, the bikes it puts into the network in the month (pointing down) or takes out of it (pointing up), each with its bikes: solid, what a plant assembles or a local warehouse delivers; dashed, what a node takes from its stock or a warehouse puts into it. Place the pointer on a node to read its kind and how its net supply is made. Under the figure, a table gives, for each month, the transport cost of the solution of the problem and that of the plan of M3, and whether the two have the same flow on every arc.

`show_tables`: instead of the figure, three tables: the net supply and the kind of each node in each month, and the flow on each arc in the solution of the problem of each month; then the same table of transport costs.

With the figure of the next cell, or with its tables, answer: in which months is the central warehouse a demand node, a transhipment node or a supply node, and why? In which months does the solution of the minimum cost flow problem give the flows of the plan of M3?

In [ ]:
# @title Minimum cost flow problem of one month { display-mode: "form" }
show_tables = False  # @param {type:"boolean"}
dn.month_flows(show_tables=show_tables);

The [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/distribution_network.md) explains all the numbers and has a script that prints them.